# Clases externa e interna

En este ejemplo, `Interna` se define dentro de `Externa`. Cada vez que se crea un objeto `Externa`, su constructor (`__init__`) instancia automáticamente un objeto `Interna` y lo guarda en `interna_por_defecto`.

La clase interna organiza comportamiento relacionado con la externa, pero sus objetos siguen siendo instancias independientes. Para acceder a uno se usa `objeto_externo.interna_por_defecto`.

In [8]:
class Externa:
    class Interna:
        def __init__(self, nombre_interno):
            self.nombre_interno = nombre_interno

        def mostrar_detalles(self):
            return f"Soy la clase interna. Mi nombre es: {self.nombre_interno}"

    def __init__(self, nombre_externo):
        self.nombre_externo = nombre_externo
        self.interna_por_defecto = self.Interna("Instancia automática")

    def mostrar_detalles(self):
        return f"Soy la clase externa. Mi nombre es: {self.nombre_externo}"

## Crear y utilizar los objetos

Al instanciar `Externa`, se crea también la instancia interna por defecto. Se puede comprobar llamando a los métodos de ambos objetos. También podemos crear otra instancia interna manualmente con un nombre diferente.

In [9]:
objeto_externo = Externa("Principal")

print(objeto_externo.mostrar_detalles())
print(objeto_externo.interna_por_defecto.mostrar_detalles())

otra_interna = Externa.Interna("Creada manualmente")
print(otra_interna.mostrar_detalles())

Soy la clase externa. Mi nombre es: Principal
Soy la clase interna. Mi nombre es: Instancia automática
Soy la clase interna. Mi nombre es: Creada manualmente


## Crear una instancia interna desde fuera de la externa

Como `Interna` está definida dentro de `Externa`, desde fuera se referencia usando `Externa.Interna(...)`. Esta forma crea directamente un objeto interno: no hace falta crear primero una instancia externa y no se ejecuta el `__init__` de `Externa`.

In [10]:
objeto_interno_directo = Externa.Interna("Instancia manual")
print(objeto_interno_directo.mostrar_detalles())

Soy la clase interna. Mi nombre es: Instancia manual


## Ejercicio resuelto: pedido y sus estados

Una clase anidada también sirve para agrupar valores que solo tienen sentido junto a la clase externa. Aquí `Estado` define los estados posibles de un `Pedido`. Como está basada en `Enum`, se seleccionan opciones predefinidas en lugar de escribir textos arbitrarios. Desde fuera se accede a ellas con `Pedido.Estado.PENDIENTE`.

In [11]:
from enum import Enum


class Pedido:
    class Estado(Enum):
        PENDIENTE = "Pendiente"
        EN_PREPARACION = "En preparación"
        ENVIADO = "Enviado"
        ENTREGADO = "Entregado"
        CANCELADO = "Cancelado"

    def __init__(self, numero):
        self.numero = numero
        self.estado = self.Estado.PENDIENTE

    def cambiar_estado(self, nuevo_estado):
        if not isinstance(nuevo_estado, self.Estado):
            raise TypeError("El estado debe ser uno de los valores de Pedido.Estado.")
        self.estado = nuevo_estado

    def __str__(self):
        return f"Pedido {self.numero}: {self.estado.value}"

Creamos un pedido y cambiamos su estado desde fuera de la clase. Observa que las opciones se indican mediante `Pedido.Estado`, sin crear primero un objeto `Estado` por separado.

In [12]:
pedido = Pedido("PED-1001")
print(pedido)

pedido.cambiar_estado(Pedido.Estado.EN_PREPARACION)
print(pedido)

pedido.cambiar_estado(Pedido.Estado.ENVIADO)
print(pedido)

pedido.cambiar_estado(Pedido.Estado.ENTREGADO)
print(pedido)

Pedido PED-1001: Pendiente
Pedido PED-1001: En preparación
Pedido PED-1001: Enviado
Pedido PED-1001: Entregado


## Ejercicio resuelto 2: Coche y Motor (composición y delegación)

En este ejemplo, la clase externa `Coche` contiene una clase interna `Motor`. Al instanciar un coche, su constructor crea automáticamente una instancia interna de `Motor`. El coche delega operaciones en el motor y permite también sustituirlo por otra instancia creada desde fuera con `Coche.Motor(...)`.

In [13]:
class Coche:
    class Motor:
        def __init__(self, tipo: str, potencia_cv: int):
            self.tipo = tipo
            self.potencia_cv = potencia_cv
            self.encendido = False

        def arrancar(self) -> str:
            if not self.encendido:
                self.encendido = True
                return f"Motor {self.tipo} ({self.potencia_cv} CV) arrancado."
            return "El motor ya estaba en marcha."

        def apagar(self) -> str:
            if self.encendido:
                self.encendido = False
                return "Motor apagado."
            return "El motor ya estaba apagado."

        def __str__(self) -> str:
            estado = "Encendido" if self.encendido else "Apagado"
            return f"Motor {self.tipo} de {self.potencia_cv} CV [{estado}]"

    def __init__(self, marca: str, modelo: str, tipo_motor: str, potencia_cv: int):
        self.marca = marca
        self.modelo = modelo
        self.motor = self.Motor(tipo_motor, potencia_cv)

    def arrancar_coche(self) -> str:
        return f"{self.marca} {self.modelo} -> {self.motor.arrancar()}"

    def apagar_coche(self) -> str:
        return f"{self.marca} {self.modelo} -> {self.motor.apagar()}"

    def cambiar_motor(self, nuevo_motor):
        if not isinstance(nuevo_motor, self.Motor):
            raise TypeError("El nuevo motor debe ser una instancia de Coche.Motor.")
        self.motor = nuevo_motor
        return f"Nuevo motor instalado con éxito en el {self.marca} {self.modelo}."

    def __str__(self) -> str:
        return f"Coche: {self.marca} {self.modelo} | {self.motor}"

Comprobamos el funcionamiento: arranque, inspección del motor interno y sustitución por un motor nuevo.

In [14]:
mi_coche = Coche("Ford", "Focus", "Gasolina", 125)
print(mi_coche)
print(mi_coche.arrancar_coche())

# Crear un motor independiente desde fuera y sustituirlo
motor_electrico = Coche.Motor("Eléctrico", 204)
print(f"Motor de repuesto: {motor_electrico}")

mi_coche.cambiar_motor(motor_electrico)
print(mi_coche)


Coche: Ford Focus | Motor Gasolina de 125 CV [Apagado]
Ford Focus -> Motor Gasolina (125 CV) arrancado.
Motor de repuesto: Motor Eléctrico de 204 CV [Apagado]
Coche: Ford Focus | Motor Eléctrico de 204 CV [Apagado]


## Referencia de la clase interna a la instancia externa

Al crear la clase interna, pasamos `self` de la clase externa para que `Cabeza` pueda acceder a los datos de `Humano`.

### Enunciado del ejercicio

1. Crea una clase externa `Humano` cuyo constructor reciba un nombre y lo guarde en `self.nombre`.
2. Define dentro de `Humano` una clase interna llamada `Cabeza`.
3. Al crear un `Humano`, instancia su `Cabeza` y pásale la referencia a la instancia externa (`self`). Guarda la cabeza en `self.cerebro`.
4. El constructor de `Cabeza` debe guardar la referencia recibida en `self.humano_referencia`.
5. Añade a `Cabeza` un método `pensar()` que devuelva un mensaje con el nombre del humano.
6. Crea una persona llamada Carlos y muestra el resultado de llamar a `persona.cerebro.pensar()`.

In [15]:
class Humano:
    def __init__(self, nombre):
        self.nombre = nombre
        # Pasamos la instancia externa al constructor de Cabeza.
        self.cerebro = self.Cabeza(self)

    class Cabeza:
        def __init__(self, humano_referencia):
            self.humano_referencia = humano_referencia

        def pensar(self):
            return f"{self.humano_referencia.nombre} está pensando intensamente."

persona = Humano("Carlos")
print(persona.cerebro.pensar())

Carlos está pensando intensamente.


## Factura con una clase interna

En este ejemplo, `Pedido` contiene la clase `Item` y crea sus instancias al agregar productos. Cada línea calcula su subtotal y el pedido genera la factura con el total.

### Enunciado del ejercicio

1. Crea una clase `Pedido` que reciba un identificador y el nombre del cliente, y prepare una lista vacía para sus artículos.
2. Define dentro de `Pedido` la clase interna `Item`, con atributos para el producto, la cantidad y el precio unitario.
3. Añade a `Item` un método que calcule el subtotal multiplicando la cantidad por el precio unitario.
4. Añade a `Pedido` un método `agregar_producto()` que cree un `Item` y lo añada a la lista del pedido.
5. Implementa un método para calcular el total sumando los subtotales y otro para mostrar la factura con el identificador, el cliente, el detalle de los artículos y el total.
6. Prueba el pedido 1024 de Ana Martínez con un portátil Asus, dos ratones Logitech y cuatro alfombrillas de ratón; muestra la factura.

In [16]:
class Pedido:
    def __init__(self, id_pedido, cliente):
        self.id_pedido = id_pedido
        self.cliente = cliente
        self.items = []

    def agregar_producto(self, producto, cantidad, precio_unitario):
        nuevo_item = self.Item(producto, cantidad, precio_unitario)
        self.items.append(nuevo_item)

    def calcular_total_pedido(self):
        return sum(item.calcular_subtotal() for item in self.items)

    def mostrar_factura(self):
        print(f"Factura ID: {self.id_pedido} | Cliente: {self.cliente}")
        print("-" * 40)
        for item in self.items:
            print(f"- {item.producto} x{item.cantidad}: {item.calcular_subtotal():.2f}€")
        print("-" * 40)
        print(f"TOTAL: {self.calcular_total_pedido():.2f}€\n")

    class Item:
        def __init__(self, producto, cantidad, precio_unitario):
            self.producto = producto
            self.cantidad = cantidad
            self.precio_unitario = precio_unitario

        def calcular_subtotal(self):
            return self.cantidad * self.precio_unitario

mi_pedido = Pedido(id_pedido=1024, cliente="Ana Martínez")
mi_pedido.agregar_producto("Portátil Asus", 1, 899.99)
mi_pedido.agregar_producto("Ratón Logitech", 2, 5.50)
mi_pedido.agregar_producto("Alfombrilla ratón", 4, 3.00)

mi_pedido.mostrar_factura()

Factura ID: 1024 | Cliente: Ana Martínez
----------------------------------------
- Portátil Asus x1: 899.99€
- Ratón Logitech x2: 11.00€
- Alfombrilla ratón x4: 12.00€
----------------------------------------
TOTAL: 922.99€



## Ejercicio: partido de tenis

La clase externa `PartidoTenis` guarda los jugadores, el estado del partido y el historial de sets. Cada set es una instancia de la clase interna `Set`, que conserva una referencia al partido para impedir que se anoten puntos si se ha suspendido por lluvia.

### Enunciado literal

1. Crea una clase externa llamada `PartidoTenis` que reciba los nombres de los jugadores (`jugador1` y `jugador2`) en su constructor. Debe inicializar un atributo `self.cancelado = False` y una lista vacia para almacenar los sets (`self.historial_sets`).
2. Al inicializar el partido, este debe generar automaticamente el primer set instanciando la clase interna `Set` y anadiendolo a `self.historial_sets`.
3. Al inicializarse el set, debe recibir la referencia al partido padre (`self.partido`) y el numero de set (por ejemplo, 1). Tendra atributos para el marcador: `self.juegos_j1 = 0` y `self.juegos_j2 = 0`.
4. La clase interna debe tener un metodo `registrar_punto(self, jugador_num)`. Antes de sumar nada, debe verificar si el partido padre sigue activo (`not self.partido.cancelado`). Si esta activo, suma un juego al jugador correspondiente. Si el partido fue cancelado, debe bloquear el marcador.
5. La clase externa tendra un metodo `anotar_en_set_actual(self, jugador_num)` que delegue la accion en el ultimo set de la lista, y un metodo `suspender_por_lluvia(self)` que cambie `self.cancelado` a `True`.


### 1. La clase externa y su estado

`PartidoTenis` representa el partido completo. El constructor guarda los nombres, empieza con `cancelado = False` y prepara `historial_sets`. Inmediatamente crea el primer set, pasando `self` para que el set conozca el partido al que pertenece.

`anotar_en_set_actual` primero comprueba que el historial no este vacio; asi evita intentar acceder a `[-1]` cuando no hay ningun set. Si lo hay, delega la anotacion al ultimo set. `suspender_por_lluvia` avisa y cambia el estado a cancelado.

### 2. La clase interna y la validacion del marcador

`_Set` se define dentro de `PartidoTenis` porque pertenece conceptualmente a un partido. El guion bajo inicial comunica que es una clase de uso interno. Recibe y guarda una referencia al partido padre, el numero del set y los dos contadores, que comienzan en cero.

Antes de sumar, `registrar_punto` comprueba que existe el partido y que no esta cancelado. Si no esta activo, imprime que deniega el punto y sale con `return`, sin modificar el marcador. Si esta activo, acepta solo `jugador_num` igual a `1` o `2`; cualquier otro valor genera `ValueError`. Despues de una anotacion valida, imprime el marcador junto con los nombres.

In [21]:
class PartidoTenis:
    def __init__(self, jugador1, jugador2):
        self.jugador1 = jugador1
        self.jugador2 = jugador2
        self.cancelado = False
        self.historial_sets = []

        primer_set = self._Set(self, 1)
        self.historial_sets.append(primer_set)

    def anotar_en_set_actual(self, jugador_num):
        if self.historial_sets:
            set_actual = self.historial_sets[-1]
            set_actual.registrar_punto(jugador_num)
        else:
            print("[Error] No hay ningun set activo en juego.")

    def suspender_por_lluvia(self):
        print("\n[Torneo] Tormenta detectada. Partido SUSPENDIDO.")
        self.cancelado = True

    class _Set:
        def __init__(self, partido_padre, numero_set):
            self.partido = partido_padre
            self.numero_set = numero_set
            self.juegos_j1 = 0
            self.juegos_j2 = 0

        def registrar_punto(self, jugador_num):
            if not self.partido or self.partido.cancelado:
                print(f"[Arbitro] Punto denegado en Set {self.numero_set}: El partido no esta activo.")
                return

            if jugador_num == 1:
                self.juegos_j1 += 1
            elif jugador_num == 2:
                self.juegos_j2 += 1
            else:
                raise ValueError("jugador_num debe ser 1 o 2.")

            print(
                f"[Set {self.numero_set}] Marcador actualizado: "
                f"{self.partido.jugador1} [{self.juegos_j1}] vs "
                f"[{self.juegos_j2}] {self.partido.jugador2}"
            )


### Que comprueba la demostracion

1. Creamos un partido que empieza activo: `cancelado` vale `False`. El constructor tambien crea automaticamente el set 1.
2. Las llamadas a `anotar_en_set_actual(1)` y `(2)` se delegan al ultimo set. Cada una incrementa el contador del jugador correspondiente e imprime el marcador actualizado.
3. `suspender_por_lluvia()` muestra el aviso del torneo y pone `cancelado` en `True`.
4. Al intentar anotar otra vez, `registrar_punto` detecta que el partido ya no esta activo. Imprime que deniega el punto y regresa sin tocar el marcador.
5. La comprobacion final del marcador permite ver que sigue igual despues de la suspension.

En este ejercicio, cada anotacion suma un juego al contador del jugador; no se implementa la puntuacion reglamentaria de tenis (15, 30, 40).


In [22]:
partido = PartidoTenis("Carlos Alcaraz", "Jannik Sinner")

# El partido esta activo: cada llamada actualiza el marcador.
partido.anotar_en_set_actual(1)
partido.anotar_en_set_actual(2)
partido.anotar_en_set_actual(1)

set_actual = partido.historial_sets[-1]
print(f"Marcador antes de la suspension: {set_actual.juegos_j1} - {set_actual.juegos_j2}")

# La suspension cambia el estado y bloquea nuevas anotaciones.
partido.suspender_por_lluvia()
partido.anotar_en_set_actual(1)

print(f"Marcador despues del intento: {set_actual.juegos_j1} - {set_actual.juegos_j2}")


[Set 1] Marcador actualizado: Carlos Alcaraz [1] vs [0] Jannik Sinner
[Set 1] Marcador actualizado: Carlos Alcaraz [1] vs [1] Jannik Sinner
[Set 1] Marcador actualizado: Carlos Alcaraz [2] vs [1] Jannik Sinner
Marcador antes de la suspension: 2 - 1

[Torneo] Tormenta detectada. Partido SUSPENDIDO.
[Arbitro] Punto denegado en Set 1: El partido no esta activo.
Marcador despues del intento: 2 - 1


### Casos de prueba adicionales

Estas pruebas verifican el estado inicial, el marcador, la validacion del jugador, la delegacion al ultimo set, el bloqueo tras cancelar el partido y el caso defensivo de un historial vacio.

### Por que usamos `assert` en las pruebas

Cada `assert condicion` comprueba una expectativa: si se cumple, la celda sigue; si no, Python lanza `AssertionError` y la prueba falla. Para el jugador invalido, esperamos especificamente `ValueError`; el bloque `try/except` confirma que se produce y luego el `assert` comprueba que el marcador quedo intacto. Para una anotacion rechazada por cancelacion, la solucion informa mediante un mensaje y no cambia el marcador, por lo que la prueba compara el estado anterior y posterior.

In [23]:
# Caso 1: se crea el primer set y el partido empieza activo.
partido_prueba = PartidoTenis("Eva", "Marco")
assert partido_prueba.cancelado is False
assert len(partido_prueba.historial_sets) == 1
primer_set = partido_prueba.historial_sets[0]
assert primer_set.numero_set == 1
assert (primer_set.juegos_j1, primer_set.juegos_j2) == (0, 0)
print("Caso 1 correcto: partido activo y primer set con marcador 0 - 0.")

# Caso 2: las anotaciones se suman al jugador correspondiente.
partido_prueba.anotar_en_set_actual(1)
partido_prueba.anotar_en_set_actual(2)
partido_prueba.anotar_en_set_actual(1)
assert (primer_set.juegos_j1, primer_set.juegos_j2) == (2, 1)
print("Caso 2 correcto: el marcador queda 2 - 1.")

# Caso 3: se rechaza un numero de jugador invalido sin cambiar el marcador.
marcador_antes = (primer_set.juegos_j1, primer_set.juegos_j2)
try:
    partido_prueba.anotar_en_set_actual(3)
except ValueError:
    pass
else:
    raise AssertionError("Se esperaba ValueError para jugador_num=3.")
assert (primer_set.juegos_j1, primer_set.juegos_j2) == marcador_antes
print("Caso 3 correcto: la entrada invalida no altera el marcador.")

# Caso 4: anotar_en_set_actual actua sobre el ultimo set del historial.
segundo_set = partido_prueba._Set(partido_prueba, 2)
partido_prueba.historial_sets.append(segundo_set)
partido_prueba.anotar_en_set_actual(2)
assert (segundo_set.juegos_j1, segundo_set.juegos_j2) == (0, 1)
assert (primer_set.juegos_j1, primer_set.juegos_j2) == (2, 1)
print("Caso 4 correcto: solo se actualiza el ultimo set.")

# Caso 5: un partido suspendido deniega el punto y conserva el marcador.
partido_prueba.suspender_por_lluvia()
marcador_antes = (segundo_set.juegos_j1, segundo_set.juegos_j2)
partido_prueba.anotar_en_set_actual(1)
assert (segundo_set.juegos_j1, segundo_set.juegos_j2) == marcador_antes
print("Caso 5 correcto: el partido suspendido no modifica el marcador.")

# Caso 6: un historial vacio se gestiona sin intentar acceder a [-1].
partido_sin_sets = PartidoTenis("Nora", "Pablo")
partido_sin_sets.historial_sets.clear()
partido_sin_sets.anotar_en_set_actual(1)
assert partido_sin_sets.historial_sets == []
print("Caso 6 correcto: el historial vacio se gestiona sin error.")


Caso 1 correcto: partido activo y primer set con marcador 0 - 0.
[Set 1] Marcador actualizado: Eva [1] vs [0] Marco
[Set 1] Marcador actualizado: Eva [1] vs [1] Marco
[Set 1] Marcador actualizado: Eva [2] vs [1] Marco
Caso 2 correcto: el marcador queda 2 - 1.
Caso 3 correcto: la entrada invalida no altera el marcador.
[Set 2] Marcador actualizado: Eva [0] vs [1] Marco
Caso 4 correcto: solo se actualiza el ultimo set.

[Torneo] Tormenta detectada. Partido SUSPENDIDO.
[Arbitro] Punto denegado en Set 2: El partido no esta activo.
Caso 5 correcto: el partido suspendido no modifica el marcador.
[Error] No hay ningun set activo en juego.
Caso 6 correcto: el historial vacio se gestiona sin error.
